In [6]:
from pathlib import Path

DATA = Path("../data")
OUT = Path("../output")

OUT.mkdir(exist_ok=True)

for file in sorted(DATA.iterdir()):
    print(f"{file.name} {file.stat().st_size / 1024:8.1f} KB")

accounts.csv     55.6 KB
monthly_usage.csv    560.3 KB
pricing_rules.json      0.5 KB
renewal_pipeline_extract.txt     11.5 KB
subscriptions.csv    170.2 KB


In [7]:
from datetime import date

import csv
import json

AS_OF = date(2026, 9, 30)

In [8]:
with open(DATA/"pricing_rules.json") as file:
    pricing = json.load(file)

print(list(pricing.keys()))

['currency', 'list_price_per_seat_per_year', 'tier_multiplier', 'segment_discount_pct', 'partner_extra_discount_pct', 'max_discount_pct', 'renewal_uplift_pct']


In [9]:
def quote_acv(product, tier, seats, segment, channel="Direct"):
    """
    Return the Annual Contract Value (USD) for a quote, using pricing_rules.json
    """

    if seats <= 0:
        raise ValueError(f"Seats must be positive, got : {seats}")

    list_price = pricing["list_price_per_seat_per_year"][product] * \
        pricing["tier_multiplier"][tier]
    discount = pricing["segment_discount_pct"][segment]

    if channel == "Partner":
        discount += pricing["partner_extra_discount_pct"]

    discount = min(discount, pricing["max_discount_pct"])
    return round(seats * list_price * (1-discount/100), 2)

In [11]:
print(quote_acv("Corvana SecureEdge", "Advantage", 500, "Mid-Market"))
print(quote_acv("Corvana SecureEdge", "Advantage", 500, "Mid-Market", channel= "Partner"))

142560.0
136080.0


In [ ]:
try:
    quote_acv("Corvana Observe", "Premier", 0, "SMB")
except ValueError as error:
    print("Value Error : ", error)

try:
    quote_acv("Corvana Voice", "Premier", 10, "SMB")
except KeyError as error:
    print("Key Error : Unknown Product", error)

Value Error :  Seats must be positive, got : 0
Key Error : Unknown Product 'Corvana Voice'


In [13]:
def urgency(renewal_date, as_of=AS_OF):
    days = (renewal_date - as_of).days

    if days < 0:
        return "Overdue"
    elif days <= 30:
        return "Critical"
    elif days <= 90:
        return "Upcoming"

    return "Later"

In [14]:
urgency_result = urgency(date(2026, 9, 1))

print(urgency_result)

Overdue


In [15]:
clean = []
rejects = []
header = None

with open(DATA/"renewal_pipeline_extract.txt") as file:
    for line_no, line in enumerate(file, start=1):
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        fields = line.split("|")

        if header is None:
            header = fields
            continue

        try:
            if len(fields) != len(header):
                raise ValueError(f"Expected {len(header)} fields, got : {len(fields)}")

            row = dict(zip(header, fields))
            row["seats"] = int(row["seats"])

            if row["seats"] <= 0:
                raise ValueError(f"Seats must be positive, got : {row['seats']}")

            row["acv_usd"] = float(row["acv_usd"])
            row["renewal_date"] = date.fromisoformat(row["renewal_date"])

            clean.append(row)
        except ValueError as error:
            rejects.append((line_no, str(error)))

print(f"Clean rows : {len(clean)}")
print(f"Rejected rows : {len(rejects)}")

Clean rows : 115
Rejected rows : 5


In [16]:
with open(OUT/"renewal_pipeline_clean.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=header)
    writer.writeheader()
    writer.writerows(clean)

with open(OUT/"renewal_pipeline_errors.log", "w") as file:
    for line_no, reason in rejects:
        file.write(f"Line {line_no} : {reason}\n")

print(f"Wrote {len(clean)} rows to {OUT/'renewal_pipeline_clean.csv'}")
print(f"Wrote {len(rejects)} rejects to {OUT/'renewal_pipeline_errors.log'}")

Wrote 115 rows to ..\output\renewal_pipeline_clean.csv
Wrote 5 rejects to ..\output\renewal_pipeline_errors.log
